# KulitKu — non_aug_v1, data tidak seimbang
Notebook Colab mandiri: **tidak perlu clone GitHub atau paket `kulitku_ml`**. Jalankan sel dari atas ke bawah. Pilih **Runtime → Change runtime type → T4 GPU** untuk ekstraksi fitur.

**Data:** `/content/drive/MyDrive/KulitKu/datasets/non_aug_v1`, empat folder Eczema, Herpes Zoster, Normal, Ringworm. Semua gambar valid tanpa awalan `aug_` digunakan; tidak ada target 1.000 atau downsampling. Jumlah aktual dihitung, bukan diasumsikan. Riwayat lama pernah menunjukkan 1.544/163/170/113 file; isi folder sekarang yang menentukan.

**Metode:** 3 CNN pretrained beku × Logistic Regression/SVM/Random Forest sklearn = 9 kombinasi. `class_weight='balanced'` pada semua kandidat. Tidak ada epoch/loss CNN karena CNN tidak dilatih ulang. Tersedia grafik macro-F1 training–validation, confusion matrix, dan probabilitas terkalibrasi.

**Pembagian:** sekitar 80% development dan 20% test per kelompok (jumlah gambar dapat berbeda karena ukuran kelompok). Development memakai 5-fold CV: di setiap putaran sekitar 64% seluruh data untuk training dan 16% untuk validation. Validation bergiliran, bukan folder holdout ketiga. Test terkunci, hanya model terpilih yang diuji. Jangan pilih ulang model berdasarkan test.

**Kebocoran:** duplikat piksel dibuang; nama keluarga file dan kemiripan pHash dikelompokkan sebelum split. Kelompok mirip digabung secara konservatif, bukan dihapus sebagai duplikat. Ini belum membuktikan pasien berbeda. Semua hasil tetap **eksploratori**. Foto tidak berawalan `aug_` belum tentu foto asli unik.

**Hasil:** Drive/KulitKu/hasil_non_aug_v1/<RUN_ID>. Dataset sumber tidak diubah. Model akhir `.joblib` + bobot CNN + metadata dikemas ZIP, beserta kode prediksi. Ini belum APK. Ekspor ONNX opsional di bagian akhir dan harus lulus uji kesetaraan.

## 1. Instal dependensi
Jalankan sekali pada runtime baru. Jika Colab meminta restart setelah instalasi, restart lalu jalankan dari sel 2.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'timm==1.0.22', 'scikit-learn==1.7.2', 'ImageHash==4.3.2',
    'pandas>=2.2,<3', 'matplotlib>=3.8,<4', 'joblib>=1.4,<2',
    'tqdm>=4.66,<5'], check=True)

## 2. Drive dan konfigurasi
Path dataset sudah diisi. Untuk melanjutkan run yang terputus, isi `RESUME_RUN_ID` dengan ID hasil sebelumnya. Jangan memakai ID eksperimen lama dari notebook berbeda.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, random, re, gc, shutil, time, warnings, subprocess
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
drive.mount('/content/drive')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import imagehash, joblib, torch, timm, sklearn
from tqdm.auto import tqdm
from IPython.display import display

DATA_ROOT = Path('/content/drive/MyDrive/KulitKu/datasets/non_aug_v1')
OUTPUT_ROOT = Path('/content/drive/MyDrive/KulitKu/hasil_non_aug_v1')
RESUME_RUN_ID = ''
# CSV opsional: relative_path,group_id. ID harus unik lintas semua kelas.
GROUPS_CSV = ''
CLASSES = ['Eczema', 'Herpes Zoster', 'Normal', 'Ringworm']
SEED = 42
PHASH_DISTANCE = 4  # Kandidat kemiripan; bukan bukti foto/pasien sama.
MAX_PIXELS = 25_000_000
BATCH_SIZE = 16
EXTRACTORS = {
    'mobilenetv4': 'mobilenetv4_conv_small.e2400_r224_in1k',
    'resnet50': 'resnet50.a1_in1k',
    'convnext_tiny': 'convnext_tiny.fb_in1k',
}
# Default menjalankan seluruh 9 kombinasi.
RUN_ID = RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT = OUTPUT_ROOT / RUN_ID
for folder in ['audit','tables','figures','features','models','export']:
    (OUT / folder).mkdir(parents=True, exist_ok=True)
CONFIG = dict(protocol='non_aug_grouped_v1', dataset_root=str(DATA_ROOT), classes=CLASSES,
              seed=SEED, phash_distance=PHASH_DISTANCE, groups_csv=GROUPS_CSV,
              max_pixels=MAX_PIXELS, extractors=EXTRACTORS, holdout_folds=5, cv_folds=5,
              class_weight='balanced', exploratory=True, patient_independence_verified=False)
def save_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    tmp.replace(path)
def digest(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()
if (OUT/'config.json').exists():
    if json.loads((OUT/'config.json').read_text()) != CONFIG:
        raise ValueError('Konfigurasi berbeda. Kosongkan RESUME_RUN_ID untuk run baru.')
else: save_json(CONFIG, OUT/'config.json')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
Image.MAX_IMAGE_PIXELS = MAX_PIXELS
assert DATA_ROOT.is_dir(), f'Folder tidak ditemukan: {DATA_ROOT}'
print('RUN_ID:', RUN_ID, '\nData:', DATA_ROOT, '\nHasil:', OUT, '\nDevice:', DEVICE)
print('Jumlah kelas dihitung pada sel berikut, tidak diubah menjadi 1.000.')
freeze = subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True,check=True).stdout
(OUT/'requirements_snapshot.txt').write_text('\n'.join(l for l in freeze.splitlines()
    if '://' not in l and ' @ ' not in l and not l.startswith('-e '))+'\n')

## 3. Inventaris, gambar rusak, dan duplikat piksel
Baca setiap file sekali dari Drive. Mungkin beberapa menit. Alpha dikomposit pada putih, orientasi EXIF diperbaiki. Duplikat yang labelnya bertentangan dikarantina otomatis, bukan dipilih salah satunya.

In [ ]:
EXT = {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff','.jfif'}
def load_rgb(path):
    with Image.open(path) as im:
        if im.width * im.height > MAX_PIXELS: raise ValueError('Melebihi batas piksel')
        im = ImageOps.exif_transpose(im)
        if im.mode in ('RGBA','LA') or 'transparency' in im.info:
            rgba=im.convert('RGBA'); bg=Image.new('RGBA',rgba.size,'white')
            return Image.alpha_composite(bg,rgba).convert('RGB')
        return im.convert('RGB')
rows=[]; rejected=[]; inventory=[]
for label in CLASSES:
    folder=DATA_ROOT/label
    if not folder.is_dir(): raise FileNotFoundError(f'Folder kelas tidak ada: {folder}')
    paths=sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in EXT)
    n_aug=0
    for p in tqdm(paths,desc=label):
        rel=p.relative_to(DATA_ROOT).as_posix()
        if p.name.lower().startswith('aug_'):
            n_aug+=1; rejected.append(dict(relative_path=rel,label=label,reason='awalan_aug')); continue
        try:
            image=load_rgb(p)
            px=hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
            rows.append(dict(relative_path=rel,label=label,pixel_hash=px,
                phash=str(imagehash.phash(image)),width=image.width,height=image.height))
        except Exception as e: rejected.append(dict(relative_path=rel,label=label,reason=str(e)))
    inventory.append(dict(label=label,total_files=len(paths),awalan_aug=n_aug))
raw=pd.DataFrame(rows).sort_values('relative_path').reset_index(drop=True)
if raw.empty: raise ValueError('Tidak ada gambar valid.')
conflict_hash=raw.groupby('pixel_hash').label.nunique().loc[lambda x:x>1].index
raw[raw.pixel_hash.isin(conflict_hash)].to_csv(OUT/'audit/conflicting_exact_labels.csv',index=False)
clean=raw[~raw.pixel_hash.isin(conflict_hash)].copy()
duplicates=clean[clean.duplicated('pixel_hash',keep='first')]
duplicates.to_csv(OUT/'audit/exact_duplicates_removed.csv',index=False)
df=clean.drop_duplicates('pixel_hash').sort_values('relative_path').reset_index(drop=True)
pd.DataFrame(rejected,columns=['relative_path','label','reason']).to_csv(OUT/'audit/rejected.csv',index=False)
raw.to_csv(OUT/'audit/inventory_valid.csv',index=False)
summary=pd.DataFrame(inventory).set_index('label')
summary['valid_sebelum_dedup']=raw.groupby('label').size()
summary['unik_piksel']=df.groupby('label').size()
display(summary.fillna(0).astype(int)); summary.to_csv(OUT/'tables/data_counts.csv')
if not set(CLASSES).issubset(set(df.label)): raise ValueError('Ada kelas kosong setelah audit.')
print('Tanpa awalan aug_ belum membuktikan foto asli unik atau pasien berbeda.')

## 4. Kelompok foto mirip sebelum split
Otomatis menggabungkan kandidat pHash jarak ≤4 dan keluarga nama seperti `158_VI-shingles (9).jpg` / `53_VI-shingles (9).jpg`. Ini langkah konservatif untuk mengurangi kebocoran. Kelompok tidak boleh dipisah hanya demi menaikkan skor. Pasangan mirip tetap disimpan untuk review; metode ini bisa melewatkan crop/rotasi tertentu. CSV kelompok pasien opsional akan digabung dengan relasi otomatis.

In [ ]:
class UnionFind:
    def __init__(self,n): self.p=list(range(n))
    def find(self,a):
        while self.p[a]!=a: self.p[a]=self.p[self.p[a]]; a=self.p[a]
        return a
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a!=b: self.p[max(a,b)]=min(a,b)
u=UnionFind(len(df)); family={}; pairs=[]
for i,r in df.iterrows():
    stem=Path(r.relative_path).stem.lower()
    # Strip numeric prefix only for the known infectious-disease naming scheme.
    canonical=re.sub(r'^\d+_(?=(?:vi|fu|ba|pa)[-_])','',stem)
    if canonical!=stem:
        key=(r.label,canonical)
        if key in family: u.union(i,family[key])
        else: family[key]=i
hashes=[int(h,16) for h in df.phash]
for i in tqdm(range(len(df)),desc='Pasangan pHash'):
    for j in range(i):
        distance=(hashes[i]^hashes[j]).bit_count()
        if distance<=PHASH_DISTANCE:
            u.union(i,j)
            pairs.append(dict(path_a=df.iloc[j].relative_path,path_b=df.iloc[i].relative_path,
                              distance=distance,same_label=bool(df.iloc[i].label==df.iloc[j].label)))
if GROUPS_CSV:
    given=pd.read_csv(GROUPS_CSV,dtype=str)
    if not {'relative_path','group_id'} <= set(given.columns): raise ValueError('CSV perlu relative_path,group_id')
    if given.relative_path.duplicated().any() or given[['relative_path','group_id']].isna().any().any():
        raise ValueError('CSV kelompok mengandung path duplikat/ID kosong.')
    mapping=dict(zip(df.relative_path,df.index)); buckets={}
    for r in given.itertuples():
        if r.relative_path not in mapping: continue
        i=mapping[r.relative_path]
        if r.group_id in buckets: u.union(i,buckets[r.group_id])
        else: buckets[r.group_id]=i
    print('CSV kelompok cocok:',given.relative_path.isin(mapping).sum(),'dari',len(df),'gambar')
df['group_id']=[f'g{u.find(i):06d}' for i in range(len(df))]
# Komponen dengan label bertentangan disisihkan untuk review.
conflicting=df.groupby('group_id').label.nunique().loc[lambda s:s>1].index
quarantine=df[df.group_id.isin(conflicting)]
quarantine.to_csv(OUT/'audit/conflicting_similar_labels.csv',index=False)
df=df[~df.group_id.isin(conflicting)].reset_index(drop=True)
pd.DataFrame(pairs,columns=['path_a','path_b','distance','same_label']).to_csv(OUT/'audit/near_pairs.csv',index=False)
df.to_csv(OUT/'audit/grouped_inventory.csv',index=False)
display(df.groupby('label').agg(gambar=('relative_path','size'),kelompok=('group_id','nunique')).reindex(CLASSES))
print('Dikarantina karena label bertentangan:',len(quarantine))
print('Hasil tetap eksploratori: identitas pasien belum diverifikasi.')
# Tampilkan contoh, semua pasangan juga tersedia di audit/near_pairs.csv.
for pair in pairs[:6]:
    print(pair['path_a'],'|',pair['path_b'],'| jarak',pair['distance'])
    fig,axes=plt.subplots(1,2,figsize=(5,2))
    for ax,p in zip(axes,[pair['path_a'],pair['path_b']]):
        ax.imshow(load_rgb(DATA_ROOT/p)); ax.axis('off')
    plt.show()

## 5. Test terkunci dan 5-fold development
Pilih holdout fold pertama dengan seed tetap; tidak mencari split yang menghasilkan skor tertinggi. Tabel di bawah adalah jumlah **aktual** setelah deduplikasi/pengelompokan. Jika kelompok terlalu sedikit untuk 5-fold, tambahkan data atau ubah protokol pada run baru.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
LABEL_TO_ID={c:i for i,c in enumerate(CLASSES)}
y_all=df.label.map(LABEL_TO_ID).to_numpy()
if df.groupby('label').group_id.nunique().min()<7:
    raise ValueError('Kelompok per kelas terlalu sedikit untuk holdout + 5-fold.')
outer=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
dev_idx,test_idx=next(outer.split(np.zeros(len(df)),y_all,df.group_id))
dev=df.iloc[dev_idx].reset_index(drop=True)
test=df.iloc[test_idx].reset_index(drop=True)
cv=list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED+1).split(
    np.zeros(len(dev)),dev.label.map(LABEL_TO_ID),dev.group_id))
for tr,va in cv:
    assert set(dev.iloc[tr].group_id).isdisjoint(set(dev.iloc[va].group_id))
    if set(dev.iloc[tr].label)!=set(CLASSES) or set(dev.iloc[va].label)!=set(CLASSES):
        raise ValueError('Ada kelas hilang pada fold. Periksa ukuran kelompok, jangan pisahkan keluarga foto.')
if set(test.label)!=set(CLASSES): raise ValueError('Ada kelas tidak terwakili pada test.')
assert set(dev.group_id).isdisjoint(set(test.group_id))
dev['fold']=-1
for k,(_,va) in enumerate(cv): dev.loc[va,'fold']=k+1
dev['partition']='development'; test['partition']='test'; test['fold']=-1
ordered=pd.concat([dev,test],ignore_index=True)
identity=digest(ordered[['relative_path','label','pixel_hash','group_id','partition','fold']].to_dict('records'))
identity_path=OUT/'data_identity.json'
if identity_path.exists() and json.loads(identity_path.read_text())['sha256']!=identity:
    raise ValueError('Data/pembagian berubah. Buat run baru, jangan memakai cache lama.')
save_json(dict(sha256=identity),identity_path)
ordered.to_csv(OUT/'tables/splits.csv',index=False)
split_counts=pd.crosstab(ordered.label,ordered.partition).reindex(CLASSES)
display(split_counts); split_counts.to_csv(OUT/'tables/split_counts.csv')
fold_counts=pd.crosstab(dev.label,dev.fold).reindex(CLASSES)
display(fold_counts); fold_counts.to_csv(OUT/'tables/validation_fold_counts.csv')
fig,ax=plt.subplots(figsize=(8,4)); split_counts.plot.bar(ax=ax)
ax.set_ylabel('Jumlah gambar'); ax.set_title('Pembagian aktual setelah audit'); plt.xticks(rotation=15)
plt.tight_layout(); fig.savefig(OUT/'figures/split_counts.png',dpi=160); plt.show()

## 6. Ekstraksi fitur tiga CNN (GPU disarankan)
CNN beku, transformasi mengikuti bobot pretrained. Tidak ada augmentasi, scaler, atau pembelajaran dari test. Fitur boleh dihitung untuk test karena tahap ini tidak melakukan fit. Cache hanya dipakai jika identitas gambar, urutan, versi, dan extractor sama.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from timm.data import resolve_model_data_config, create_transform
class ImageDataset(Dataset):
    def __init__(self,frame,transform): self.frame=frame; self.transform=transform
    def __len__(self): return len(self.frame)
    def __getitem__(self,i): return self.transform(load_rgb(DATA_ROOT/self.frame.iloc[i].relative_path))
feature_paths={}
for cnn,model_id in EXTRACTORS.items():
    folder=OUT/'features'/cnn; folder.mkdir(exist_ok=True)
    path=folder/'features.npy'; key=digest(dict(data=identity,model=model_id,timm=timm.__version__,torch=torch.__version__))
    if path.exists() and (folder/'metadata.json').exists():
        meta=json.loads((folder/'metadata.json').read_text())
        if meta['identity']!=key: raise ValueError('Cache fitur berbeda, gunakan run baru.')
        if np.load(path,mmap_mode='r').shape[0]!=len(ordered): raise ValueError('Cache fitur tidak lengkap.')
        feature_paths[cnn]=path; print('Cache:',cnn); continue
    model=timm.create_model(model_id,pretrained=True,num_classes=0).eval().to(DEVICE)
    for p in model.parameters(): p.requires_grad_(False)
    preprocessing=resolve_model_data_config(model)
    transform=create_transform(**preprocessing,is_training=False)
    loader=DataLoader(ImageDataset(ordered,transform),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    chunks=[]
    with torch.inference_mode():
        for batch in tqdm(loader,desc=cnn): chunks.append(model(batch.to(DEVICE)).cpu().numpy().astype('float32'))
    features=np.concatenate(chunks)
    if not np.isfinite(features).all(): raise ValueError('Fitur mengandung NaN/Inf.')
    np.save(path,features)
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},folder/'extractor_state.pt')
    save_json(dict(identity=key,model_id=model_id,preprocessing=preprocessing,shape=list(features.shape)),folder/'metadata.json')
    feature_paths[cnn]=path
    del model,chunks,features; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 7. Training sembilan kombinasi, bobot balanced, dan checkpoint
Scaler LR/SVM di-fit hanya pada training fold melalui Pipeline. Bobot balanced dihitung estimator dari label training fold: `n_train / (4 × n_train_kelas)`. Tidak memakai jumlah validation/test. RF tanpa scaler.

Default: LR 3 kandidat, SVM 6 kandidat, RF 4 kandidat, masing-masing 5-fold untuk tiga CNN. Konvergensi gagal akan menghentikan sel; perbaiki sebelum mengubah protokol dan gunakan run baru. CV kandidat terbaik masih optimistis akibat pemilihan parameter, bukan skor test independen.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.base import clone
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, log_loss
from sklearn.exceptions import ConvergenceWarning
from sklearn.utils.class_weight import compute_class_weight
GRIDS={
 'logistic_regression': {'model__C':[0.1,1,10]},
 'svm': {'model__C':[0.1,1,10],'model__kernel':['linear','rbf'],'model__gamma':['scale']},
 'random_forest': {'model__n_estimators':[200,400],'model__max_depth':[None,20]},
}
def make_classifier(name):
    if name=='logistic_regression':
        return Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(
            class_weight='balanced',max_iter=10000,solver='lbfgs',random_state=SEED))])
    if name=='svm': return Pipeline([('scaler',StandardScaler()),('model',SVC(
        class_weight='balanced',probability=False,random_state=SEED,cache_size=512))])
    return Pipeline([('model',RandomForestClassifier(class_weight='balanced',random_state=SEED,n_jobs=2))])
y_dev=dev.label.map(LABEL_TO_ID).to_numpy(); n_dev=len(dev)
display(pd.DataFrame({'label':CLASSES,'bobot_seluruh_development':compute_class_weight(
    'balanced',classes=np.arange(len(CLASSES)),y=y_dev)}))
# Tabel bobot training setiap fold.
weight_rows=[]
for k,(tr,va) in enumerate(cv,1):
    for label,w in zip(CLASSES,compute_class_weight('balanced',classes=np.arange(4),y=y_dev[tr])):
        weight_rows.append(dict(fold=k,label=label,weight=float(w)))
pd.DataFrame(weight_rows).to_csv(OUT/'tables/class_weights_by_fold.csv',index=False)
results=[]
for cnn in EXTRACTORS:
    x=np.load(feature_paths[cnn],mmap_mode='r')[:n_dev]
    for name,param_grid in GRIDS.items():
        combo=cnn+'__'+name; folder=OUT/'models'/combo; folder.mkdir(exist_ok=True)
        grid=list(ParameterGrid(param_grid)); key=digest(dict(data=identity,features=json.loads(
            (OUT/'features'/cnn/'metadata.json').read_text())['identity'],grid=param_grid,
            estimator=make_classifier(name).get_params(),sklearn=sklearn.__version__))
        checkpoint=folder/'cv_progress.json'
        records=[]
        if checkpoint.exists():
            old=json.loads(checkpoint.read_text())
            if old['identity']!=key: raise ValueError('Checkpoint berbeda, buat run baru.')
            records=old['records']
        done={(r['candidate'],r['fold']) for r in records}
        for candidate,params in enumerate(grid):
            for k,(tr,va) in enumerate(cv,1):
                if (candidate,k) in done: continue
                print(f'{combo}: kandidat {candidate+1}/{len(grid)} fold {k}/5',flush=True)
                estimator=make_classifier(name).set_params(**params); start=time.perf_counter()
                with warnings.catch_warnings():
                    warnings.simplefilter('error',ConvergenceWarning)
                    estimator.fit(x[tr],y_dev[tr])
                pred_tr=estimator.predict(x[tr]); pred_va=estimator.predict(x[va])
                records.append(dict(candidate=candidate,fold=k,params=params,fit_seconds=time.perf_counter()-start,
                    train_macro_f1=float(f1_score(y_dev[tr],pred_tr,average='macro',labels=range(4),zero_division=0)),
                    validation_macro_f1=float(f1_score(y_dev[va],pred_va,average='macro',labels=range(4),zero_division=0)),
                    train_accuracy=float(accuracy_score(y_dev[tr],pred_tr)),validation_accuracy=float(accuracy_score(y_dev[va],pred_va))))
                save_json(dict(identity=key,records=records),checkpoint)
        table=pd.DataFrame(records); table.to_csv(folder/'cv_all_candidates.csv',index=False)
        ranking=table.groupby('candidate').agg(mean_macro_f1=('validation_macro_f1','mean'),
            std_macro_f1=('validation_macro_f1','std'),train_macro_f1=('train_macro_f1','mean')).reset_index()
        ranking=ranking.sort_values(['mean_macro_f1','candidate'],ascending=[False,True])
        winner=int(ranking.iloc[0]['candidate']); best_params=grid[winner]
        best=make_classifier(name).set_params(**best_params)
        with warnings.catch_warnings():
            warnings.simplefilter('error',ConvergenceWarning); best.fit(x,y_dev)
        joblib.dump(best,folder/'classifier_uncalibrated.joblib')
        best_folds=table[table.candidate==winner].copy(); best_folds.to_csv(folder/'best_folds.csv',index=False)
        result=dict(combo_id=combo,cnn=cnn,classifier=name,best_params=best_params,
            mean_macro_f1=float(ranking.iloc[0].mean_macro_f1),std_macro_f1=float(ranking.iloc[0].std_macro_f1),
            train_macro_f1=float(ranking.iloc[0].train_macro_f1),exploratory=True)
        save_json(result,folder/'summary.json'); results.append(result)
        print('Selesai:',combo,result['mean_macro_f1'])
overview=pd.DataFrame(results); overview['gap']=overview.train_macro_f1-overview.mean_macro_f1
overview.to_csv(OUT/'tables/cv_overview.csv',index=False)
display(overview[['combo_id','train_macro_f1','mean_macro_f1','std_macro_f1','gap']])

## 8. Grafik training–validation dan pilihan model terkunci
Model dipilih berdasarkan mean macro-F1 CV tertinggi; jika sama persis, combo_id menjadi pemecah seri. Ini pemilihan kualitas, bukan benchmark kecepatan. Gap kecil tidak membuktikan bebas kebocoran. Skor RF lebih rendah juga tidak otomatis berarti lebih aman dari overfit.

In [ ]:
fig,ax=plt.subplots(figsize=(12,5))
pos=np.arange(len(overview)); width=.38
ax.bar(pos-width/2,overview.train_macro_f1,width,label='Training fold')
ax.bar(pos+width/2,overview.mean_macro_f1,width,yerr=overview.std_macro_f1,label='Validation fold ± SD')
ax.set_xticks(pos,overview.combo_id,rotation=70,ha='right'); ax.set_ylim(0,1.05)
ax.set_ylabel('Macro-F1'); ax.legend(); ax.set_title('Eksploratori: gap training–validation')
plt.tight_layout(); fig.savefig(OUT/'figures/train_validation_gap.png',dpi=160); plt.show()
fig,ax=plt.subplots(figsize=(11,5))
for r in overview.itertuples():
    f=pd.read_csv(OUT/'models'/r.combo_id/'best_folds.csv')
    ax.plot(f.fold,f.validation_macro_f1,marker='o',label=r.combo_id)
ax.set(xlabel='Fold',ylabel='Macro-F1 validation',title='Variasi antar fold (parameter terpilih)')
ax.legend(fontsize=7,bbox_to_anchor=(1.02,1)); plt.tight_layout()
fig.savefig(OUT/'figures/fold_variation.png',dpi=160); plt.show()
choice=overview.sort_values(['mean_macro_f1','combo_id'],ascending=[False,True]).iloc[0].to_dict()
choice['selection_rule']='Highest mean CV macro-F1, then combo_id; no test use'
selection_path=OUT/'selection.json'
if selection_path.exists():
    locked=json.loads(selection_path.read_text())
    if digest(locked)!=digest(choice): raise ValueError('Pilihan berbeda dari yang terkunci. Gunakan run baru.')
else: save_json(choice,selection_path)
selected=json.loads(selection_path.read_text()); print(json.dumps(selected,indent=2))

## 9. Kalibrasi confidence dari development saja
Sigmoid calibration memakai prediksi out-of-fold dengan kelompok CV yang sama, bukan test. Estimator dasar kemudian di-fit pada seluruh development (`ensemble=False`). Pemilihan hyperparameter sebelumnya memakai development yang sama: estimasi CV tetap memiliki bias seleksi. Kalibrasi tidak menjamin akurasi atau kepastian klinis; probabilitas hanya di antara empat kelas, bukan detektor penyakit di luar kelas.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
combo=selected['combo_id']; cnn=selected['cnn']; name=selected['classifier']
x=np.load(feature_paths[cnn],mmap_mode='r')
cal_path=OUT/'models'/combo/'classifier_calibrated.joblib'
if cal_path.exists():
    calibrated=joblib.load(cal_path)
else:
    base=make_classifier(name).set_params(**selected['best_params'])
    calibrated=CalibratedClassifierCV(estimator=base,method='sigmoid',cv=cv,ensemble=False,n_jobs=1)
    with warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning); calibrated.fit(x[:n_dev],y_dev)
    joblib.dump(calibrated,cal_path)
assert np.array_equal(calibrated.classes_,np.arange(4))
save_json(dict(method='sigmoid',cv='5-fold grouped development only',ensemble=False,
               test_used=False,classes=CLASSES),OUT/'models'/combo/'calibration.json')
print('Model terkalibrasi disimpan:',cal_path)

## 10. Final test sekali untuk model terpilih
Evaluasi model terkalibrasi yang akan diekspor. Test tidak digunakan memilih model/parameter/threshold. Jika hasil sudah ada, tampilkan hasil tersimpan. Jangan mengulang tuning berdasarkan hasil ini. Confidence interval memakai bootstrap kelompok; pada sampel kecil interval bisa lebar dan tetap belum membuktikan independensi pasien.

In [ ]:
from sklearn.metrics import precision_score, recall_score
final_dir=OUT/'final_test'; final_dir.mkdir(exist_ok=True)
metrics_path=final_dir/'metrics.json'
if metrics_path.exists():
    metrics=json.loads(metrics_path.read_text())
    report=pd.read_csv(final_dir/'per_class.csv')
    predictions=pd.read_csv(final_dir/'predictions.csv')
else:
    y_test=test.label.map(LABEL_TO_ID).to_numpy()
    probabilities=calibrated.predict_proba(x[n_dev:])
    pred=np.asarray(calibrated.classes_)[probabilities.argmax(axis=1)]
    predictions=test[['relative_path','label','group_id']].copy()
    predictions['y_true']=y_test; predictions['y_pred']=pred
    predictions['predicted_label']=[CLASSES[i] for i in pred]
    for i,c in enumerate(CLASSES): predictions['probability_'+c]=probabilities[:,i]
    predictions.to_csv(final_dir/'predictions.csv',index=False)
    raw_report=classification_report(y_test,pred,labels=range(4),target_names=CLASSES,output_dict=True,zero_division=0)
    report=pd.DataFrame([dict(label=c,**raw_report[c]) for c in CLASSES])
    report.to_csv(final_dir/'per_class.csv',index=False)
    rng=np.random.default_rng(SEED); groups=test.group_id.unique(); boot=[]; discarded=0
    members={g:np.flatnonzero(test.group_id.to_numpy()==g) for g in groups}
    for _ in range(1000):
        ix=np.concatenate([members[g] for g in rng.choice(groups,len(groups),replace=True)])
        if len(np.unique(y_test[ix]))<4: discarded+=1; continue
        boot.append(f1_score(y_test[ix],pred[ix],labels=range(4),average='macro',zero_division=0))
    ci=[float(v) for v in np.quantile(boot,[.025,.975])] if boot else None
    metrics=dict(combo_id=combo,macro_f1=float(f1_score(y_test,pred,average='macro',labels=range(4),zero_division=0)),
        accuracy=float(accuracy_score(y_test,pred)),macro_precision=float(precision_score(y_test,pred,average='macro',zero_division=0)),
        macro_recall=float(recall_score(y_test,pred,average='macro',zero_division=0)),
        log_loss=float(log_loss(y_test,probabilities,labels=range(4))),test_images=len(test),test_groups=len(groups),
        macro_f1_group_bootstrap_ci95=ci,bootstrap_valid_replicates=len(boot),discarded_missing_class_replicates=discarded,
        exploratory=True,patient_independence_verified=False,calibrated=True)
    save_json(metrics,metrics_path)
display(metrics); display(report)
y_true=predictions.y_true.to_numpy(); y_pred=predictions.y_pred.to_numpy()
cm=confusion_matrix(y_true,y_pred,labels=range(4))
pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(final_dir/'confusion_matrix.csv')
fig,ax=plt.subplots(figsize=(7,6)); ConfusionMatrixDisplay(cm,display_labels=CLASSES).plot(ax=ax,colorbar=False)
ax.set_title('Final test — model terkalibrasi (eksploratori)'); plt.xticks(rotation=20)
plt.tight_layout(); fig.savefig(OUT/'figures/final_test_confusion.png',dpi=160); plt.show()
print('Kesalahan:',int((y_true!=y_pred).sum()),'dari',len(y_true))
display(predictions[predictions.y_true!=predictions.y_pred][['relative_path','label','predicted_label']])

## 11. Paket model akhir + kode prediksi mandiri
Paket berisi bobot CNN, classifier terkalibrasi (termasuk scaler), urutan label, preprocessing, hasil test, dan manifest pembagian. Tidak menyertakan gambar dataset. Jangan memuat `.joblib` dari sumber tak dipercaya. Model yang dikemas tetap model development yang diuji; tidak di-fit ulang dengan test.

In [ ]:
PACKAGE=OUT/'export'/'kulitku_model'; PACKAGE.mkdir(parents=True,exist_ok=True)
meta=json.loads((OUT/'features'/cnn/'metadata.json').read_text())
shutil.copy2(OUT/'features'/cnn/'extractor_state.pt',PACKAGE/'extractor_state.pt')
shutil.copy2(cal_path,PACKAGE/'classifier_calibrated.joblib')
save_json(CLASSES,PACKAGE/'labels.json'); save_json(meta['preprocessing'],PACKAGE/'preprocessing.json')
save_json(dict(combo_id=combo,model_id=EXTRACTORS[cnn],data_identity=identity,classes=CLASSES,
    calibrated=True,exploratory=True,patient_independence_verified=False,test_used_for_fit=False,
    versions=dict(torch=torch.__version__,timm=timm.__version__,sklearn=sklearn.__version__,numpy=np.__version__),
    test_metrics=metrics),PACKAGE/'release.json')
shutil.copy2(OUT/'tables/splits.csv',PACKAGE/'splits.csv')
shutil.copy2(OUT/'requirements_snapshot.txt',PACKAGE/'requirements_snapshot.txt')
(PACKAGE/'predictor.py').write_text("from pathlib import Path\nimport json\nimport numpy as np\nimport torch, timm, joblib\nfrom PIL import Image, ImageOps\nfrom timm.data import create_transform\nclass Predictor:\n    def __init__(self, package_dir, device='cpu'):\n        self.root=Path(package_dir); self.device=device\n        self.release=json.loads((self.root/'release.json').read_text())\n        self.labels=json.loads((self.root/'labels.json').read_text())\n        self.pre=json.loads((self.root/'preprocessing.json').read_text())\n        self.model=timm.create_model(self.release['model_id'],pretrained=False,num_classes=0)\n        self.model.load_state_dict(torch.load(self.root/'extractor_state.pt',map_location='cpu',weights_only=True))\n        self.model.to(device).eval()\n        self.transform=create_transform(**self.pre,is_training=False)\n        self.classifier=joblib.load(self.root/'classifier_calibrated.joblib')\n        if list(self.classifier.classes_)!=list(range(len(self.labels))):\n            raise ValueError('Urutan kelas classifier berbeda dari metadata')\n    def predict(self,path):\n        with Image.open(path) as im:\n            if im.width*im.height>25_000_000: raise ValueError('Gambar terlalu besar')\n            im=ImageOps.exif_transpose(im)\n            if im.mode in ('RGBA','LA') or 'transparency' in im.info:\n                rgba=im.convert('RGBA'); im=Image.alpha_composite(Image.new('RGBA',rgba.size,'white'),rgba).convert('RGB')\n            else: im=im.convert('RGB')\n            batch=self.transform(im).unsqueeze(0).to(self.device)\n        with torch.inference_mode(): feature=self.model(batch).cpu().numpy().astype('float32')\n        p=self.classifier.predict_proba(feature)[0]\n        order=np.argsort(-p)\n        return {'label':self.labels[int(order[0])], 'probabilities':\n                [{'label':self.labels[int(i)],'percent':float(p[i]*100)} for i in order]}\n",encoding='utf-8')

(PACKAGE/'README.md').write_text(
    '# KulitKu model eksperimen\n\n'
    'Instal numpy, Pillow, torch, torchvision, timm, scikit-learn, joblib. '
    'Versi training ada di requirements_snapshot.txt.\n\n'
    '```python\nfrom predictor import Predictor\np = Predictor(".")\nprint(p.predict("foto.jpg"))\n```\n\n'
    'Empat kelas saja. Persentase bukan kepastian diagnosis. '
    'Independensi pasien belum diverifikasi. Model dilatih hanya development.\n',encoding='utf-8')
checks={}
for p in PACKAGE.iterdir():
    if p.is_file() and p.name!='checksums.json':
        h=hashlib.sha256()
        with p.open('rb') as f:
            for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
        checks[p.name]=h.hexdigest()
save_json(checks,PACKAGE/'checksums.json')
ZIP_PATH=Path(shutil.make_archive(str(OUT/'export'/('KulitKu_Model_'+RUN_ID)),'zip',root_dir=PACKAGE))
print('Paket model:',ZIP_PATH)
print('Hasil lengkap tetap di:',OUT)

## 12. Uji upload foto sendiri dengan confidence
Opsional setelah final test. Foto upload hanya untuk demo; tidak masuk training dan jangan digunakan untuk mengubah pilihan setelah melihat test. Hasil tertinggi tetap bisa salah, bahkan jika persentasenya tinggi.

In [ ]:
from google.colab import files
import importlib.util
spec=importlib.util.spec_from_file_location('kulitku_predictor',PACKAGE/'predictor.py')
module=importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
predictor=module.Predictor(PACKAGE,device=DEVICE)
uploaded=files.upload()
upload_dir=Path('/content/kulitku_demo_upload'); upload_dir.mkdir(exist_ok=True)
for i,(filename,content) in enumerate(uploaded.items()):
    p=upload_dir/f'{i}_{Path(filename).name}'; p.write_bytes(content)
    try:
        start=time.perf_counter(); result=predictor.predict(p)
        image=load_rgb(p); image.thumbnail((500,500)); display(image)
        print('File:',filename,'| Prediksi:',result['label'])
        display(pd.DataFrame(result['probabilities']))
        print(f'Waktu: {(time.perf_counter()-start)*1000:.1f} ms')
    except Exception as e: print('Gagal:',filename,str(e))

## 13. Unduh ZIP model
Sudah tersimpan permanen di Drive; unduh ini untuk menggunakan model di laptop. ZIP bukan APK dan belum dapat dipasang langsung ke HP.

In [ ]:
from google.colab import files
files.download(str(ZIP_PATH))

## 14. Ekspor ONNX opsional (CNN + classifier terkalibrasi)
Model native sudah selesai pada bagian 11. Jalankan bagian ini hanya jika diperlukan untuk integrasi runtime ONNX. Set `RUN_ONNX_EXPORT=True`; ekspor akan diuji terhadap model native pada maksimal 5 gambar development per kelas. **Test tidak dipakai untuk memilih atau mengubah model.** Jika converter tidak mendukung model atau probabilitas/label berbeda di luar toleransi, tidak dibuat ZIP ONNX siap pakai. Jangan menghapus pemeriksaan parity.

In [ ]:
RUN_ONNX_EXPORT = False
if RUN_ONNX_EXPORT:
    subprocess.run([sys.executable,'-m','pip','install','-q','onnx>=1.17,<2',
                    'onnxruntime>=1.20,<2','skl2onnx>=1.19,<2'],check=True)
    import onnxruntime as ort
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    export_dir=OUT/'export'/'onnx'; export_dir.mkdir(exist_ok=True)
    native=timm.create_model(EXTRACTORS[cnn],pretrained=False,num_classes=0).eval()
    native.load_state_dict(torch.load(PACKAGE/'extractor_state.pt',map_location='cpu',weights_only=True))
    sample=torch.zeros((1,*meta['preprocessing']['input_size']),dtype=torch.float32)
    with torch.inference_mode(): dim=int(native(sample).shape[1])
    torch.onnx.export(native,sample,str(export_dir/'extractor.onnx'),input_names=['images'],
        output_names=['features'],dynamic_axes={'images':{0:'batch'},'features':{0:'batch'}},
        opset_version=17,dynamo=False)
    converted=convert_sklearn(calibrated,initial_types=[('features',FloatTensorType([None,dim]))],
        options={id(calibrated):{'zipmap':False}},target_opset=17)
    (export_dir/'classifier_calibrated.onnx').write_bytes(converted.SerializeToString())
    sess_cnn=ort.InferenceSession(str(export_dir/'extractor.onnx'),providers=['CPUExecutionProvider'])
    sess_clf=ort.InferenceSession(str(export_dir/'classifier_calibrated.onnx'),providers=['CPUExecutionProvider'])
    samples=pd.concat([dev[dev.label==c].head(5) for c in CLASSES])
    parity=[]
    for r in samples.itertuples():
        tensor=predictor.transform(load_rgb(DATA_ROOT/r.relative_path)).unsqueeze(0)
        with torch.inference_mode(): f_native=native(tensor).numpy().astype('float32')
        f_onnx=sess_cnn.run(None,{'images':tensor.numpy()})[0]
        p_native=calibrated.predict_proba(f_native)
        outputs=sess_clf.run(None,{'features':f_onnx})
        probs=[np.asarray(v) for v in outputs if isinstance(v,np.ndarray) and v.ndim==2 and v.shape[1]==4]
        if len(probs)!=1: raise ValueError('Output probabilitas ONNX tidak dapat dipastikan.')
        p_onnx=probs[0]
        passed=bool(np.allclose(p_native,p_onnx,rtol=1e-3,atol=1e-4) and
                    np.array_equal(p_native.argmax(1),p_onnx.argmax(1)))
        parity.append(dict(relative_path=r.relative_path,max_probability_error=float(abs(p_native-p_onnx).max()),passed=passed))
    pd.DataFrame(parity).to_csv(export_dir/'parity.csv',index=False)
    if not all(r['passed'] for r in parity): raise ValueError('Parity ONNX gagal. Gunakan paket native; ekspor belum layak.')
    for filename in ['labels.json','preprocessing.json','release.json']:
        shutil.copy2(PACKAGE/filename,export_dir/filename)
    save_json(dict(cnn_input='images',cnn_output='features',classifier_input='features',
        classifier_outputs=[o.name for o in sess_clf.get_outputs()],preprocessing_implemented_in_onnx=False,
        note='Implementasikan decode EXIF/alpha dan transform timm persis seperti paket native.'),export_dir/'io.json')
    onnx_zip=shutil.make_archive(str(OUT/'export'/('KulitKu_ONNX_'+RUN_ID)),'zip',root_dir=export_dir)
    print('Parity lulus. ONNX:',onnx_zip)
else:
    print('Dilewati. Model native dan ZIP sudah dihasilkan pada bagian 11.')

## Membaca hasil dan melanjutkan sesi
- `tables/data_counts.csv`: jumlah nyata sebelum/sesudah deduplikasi; tidak harus seimbang.
- `tables/splits.csv`: setiap foto, kelompok, development/test, dan fold validation.
- `tables/cv_overview.csv`: 9 kombinasi, training/validation macro-F1 dan gap.
- `figures/train_validation_gap.png`, `fold_variation.png`: grafik evaluasi classifier, **bukan loss per epoch**.
- `final_test/per_class.csv`: periksa recall/F1 **Herpes Zoster** dan support-nya, jangan hanya accuracy.
- `final_test/metrics.json`: hasil model terpilih dan interval bootstrap kelompok.
- `export/KulitKu_Model_*.zip`: model final dan kode inferensi. ONNX opsional memerlukan integrasi aplikasi tersendiri.

Untuk melanjutkan setelah runtime terputus: jalankan notebook lagi dengan `RESUME_RUN_ID` yang sama dan konfigurasi/data sama. Fitur dan fit kandidat/fold yang selesai dipakai kembali. Audit tetap dibaca ulang untuk mendeteksi perubahan dataset. Jangan memakai RUN_ID lama jika dataset atau protokol berubah.

**Keterbatasan:** pengelompokan pHash/nama bukan verifikasi pasien. Foto dari sumber sama dapat memiliki bias watermark/latar. Kelompok zoster bisa jauh lebih sedikit dari 160 file. Probabilitas terkalibrasi tidak menyelesaikan kesalahan label, data bocor, atau pergeseran sumber. Tidak ada batas gap universal yang membuktikan bebas overfit. Simpan eksperimen ini sebagai eksploratori dan evaluasi sumber eksternal terpisah untuk langkah berikutnya.

Referensi API: [timm MobileNetV4](https://huggingface.co/timm/mobilenetv4_conv_small.e2400_r224_in1k), [sklearn calibration](https://scikit-learn.org/1.7/modules/generated/sklearn.calibration.CalibratedClassifierCV.html), [StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html).